# 12 — DeepSeek-V3.2 (Azure AI Foundry Models)

Valuta **DeepSeek-V3.2** servito come deployment serverless di Azure AI Foundry Models
(fatturazione Microsoft), con lo stesso prompt zero-shot e gli stessi parametri dei notebook
07–11. Il client è `openai.OpenAI` puntato alla **rotta OpenAI-compatibile** della risorsa
Foundry. (V3.2 sostituisce DeepSeek-V3, non più presente nel catalogo Foundry; su Azure costa
anche meno: 0,58 $/M input e 1,68 $/M output.)

Ambiti come nei notebook 10–11: `sample` (centesimi) e `test` (5.610 righe, ~12 $ — anche
DeepSeek su Foundry **non** ha una Batch API).

⚠️ DeepSeek-V3.2 è un modello **ibrido** (modalità thinking e non-thinking). Sulla rotta
chat-completions la modalità non-thinking è quella attesa di default; per sicurezza `genera()`
rimuove un eventuale blocco `<think>…</think>` in testa alla risposta. Nello smoke test
verificare che i riassunti siano testo normale e non vuoti (una risposta vuota con
`finish_reason=length` indicherebbe thinking attivo che consuma il budget da 300 token — il
caso gemma del notebook 08).

Prerequisiti: deployment serverless di DeepSeek-V3.2 nella risorsa Foundry e variabili
d'ambiente `AZURE_INFERENCE_ENDPOINT` e `AZURE_INFERENCE_API_KEY`. ⚠️ Verificare la rotta esatta
nel blade del deployment: la forma OpenAI-compatibile è
`https://<risorsa>.services.ai.azure.com/openai/v1` (se disponibile solo l'API Model Inference,
adeguare `base_url` di conseguenza).

## Ripresa e rischio di mescolare corse

⚠️ Il ciclo condiviso salta i `row_id` già presenti nel TSV di output: rieseguire la generazione
su un file esistente **aggiunge solo le righe mancanti**. È il comportamento voluto per riprendere
una corsa interrotta, ma con un modello, un deployment o una configurazione diversi si
mescolerebbero due corse nello stesso file: in quel caso **eliminare prima** il TSV e rigenerare
tutto (rieseguendo poi la valutazione). Ogni ambito (`sample`, `test`) scrive su un file separato.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import openai  # noqa: F401
except ImportError:
    %pip install openai

In [ ]:
# --- Configurazione ---------------------------------------------------------
import os
import summ_utils as su

METODO     = 'deepseek'
SCOPE      = 'sample'    # 'sample' = campione condiviso; 'test' = intera split test (5.610 righe)
N_SAMPLES  = 100
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti

MODELLO  = 'DeepSeek-V3.2'                            # nome del deployment serverless
ENDPOINT = os.environ['AZURE_INFERENCE_ENDPOINT']     # es. https://<risorsa>.services.ai.azure.com/openai/v1
API_KEY  = os.environ['AZURE_INFERENCE_API_KEY']

MAX_TOKENS  = 300
TEMPERATURE = 0.3
PROMPT_SYSTEM = ('You are a helpful assistant that summarizes news articles '
                 'from different sources concisely.')
PROMPT_USER   = 'Summarize the following document into a comprehensive summary: {documento}'
ETICHETTA   = 'DeepSeek '
NOTE_CONFIG = ('prompt zero-shot in inglese identico ai notebook 07-11; endpoint '
               'OpenAI-compatibile di Foundry Models; modello ibrido usato in modalita\' '
               'non-thinking (eventuale blocco <think> rimosso); ambiti sample e test')

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'

config = {'modello': MODELLO,
          'backend': 'Azure AI Foundry Models (endpoint OpenAI-compatibile)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE,
          'prompt_system': PROMPT_SYSTEM, 'prompt_user': PROMPT_USER,
          'note': NOTE_CONFIG}

print(f'Modello : {MODELLO} via {ENDPOINT}')
print(f'Ambito  : {SCOPE}')
print(f'Output  : {OUT_PATH}')

## Generazione dei riassunti

Stessa interfaccia chat-completions dei notebook 07–10; cambia solo `base_url`. Una risposta
vuota solleva un'eccezione, così la riga viene registrata come errore e **non** scritta nel TSV
(ritentabile alla corsa successiva).

In [ ]:
import re

from openai import OpenAI

client = OpenAI(base_url=ENDPOINT, api_key=API_KEY)

def genera(documento):
    risposta = client.chat.completions.create(
        model=MODELLO,
        messages=[{'role': 'system', 'content': PROMPT_SYSTEM},
                  {'role': 'user', 'content': PROMPT_USER.format(documento=documento)}],
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE)
    scelta = risposta.choices[0]
    contenuto = scelta.message.content or ''
    # Guardia per il modello ibrido: rimuove un eventuale blocco di reasoning in testa
    contenuto = re.sub(r'^\s*<think>.*?</think>', '', contenuto, flags=re.DOTALL).strip()
    if not contenuto:
        # solleva -> il ciclo condiviso registra l'errore e NON scrive la riga
        raise RuntimeError(f'risposta vuota (finish_reason={scelta.finish_reason})')
    return contenuto

if SCOPE == 'sample':
    esempi = su.carica_campione(SAMPLE_PATH)
elif SCOPE == 'test':
    esempi = su.itera_split(P['complete_tab'], 'test')
else:
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA)
scrittore.chiudi()

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/{metodo}_{scope}_per_example.csv` e `…_aggregate.json`.
Per l'ambito `test` i riferimenti vengono letti in streaming da `complete.tab`.

In [ ]:
import json

riassunti = su.carica_riassunti(OUT_PATH)
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

## Ispezione qualitativa

In [ ]:
if SCOPE == 'sample':
    riferimenti = su.carica_campione(SAMPLE_PATH)
else:
    riferimenti = su.itera_split(P['complete_tab'], 'test')
su.mostra_esempi(riferimenti, riassunti, quanti=2)